In [6]:
%pip install evaluate sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Token F1 Score 

Il mesure le taux de mots exacts en commun entre deux phrases sans se soucier de l'ordre des mots ou de la grammaire.

On peut l'utiliser
-   Question-Answering factuel: Lorsque la réponse attendu est un nom , une date etc
- Extraction d'informations


In [7]:
import pandas as pd
import evaluate

df = pd.read_csv("evaluation_results.csv")

metric_f1 = evaluate.load("squad")


def token_f1(prediction, ground_truth):

    predictions = [
        {
            "id": "1",
            "prediction_text": prediction
        }
    ]

    references = [
        {
            "id": "1",
            "answers": {
                "text": [ground_truth],
                "answer_start": [0]
            }
        }
    ]

    result = metric_f1.compute(
        predictions=predictions,
        references=references
    )

    return result["f1"] / 100


df["f1_without_rag"] = df.apply(
    lambda row: token_f1(
        row["llm_without_rag"],
        row["ground_truth"]
    ),
    axis=1
)

df["f1_with_rag"] = df.apply(
    lambda row: token_f1(
        row["llm_with_rag"],
        row["ground_truth"]
    ),
    axis=1
)

print(df[[
    "question",
    "f1_without_rag",
    "f1_with_rag"
]])

                                             question  f1_without_rag  \
0   ** What are the generic procedure parameters u...        0.079365   
1   If the length of the V2X service info contents...        0.102190   
2   What is the name of the attribute of the media...        0.089552   
3   What is the maximum number of simultaneous MCV...        0.290323   
4   What are the exact instants on which the value...        0.119205   
5   What is the maximum channel bandwidth for a sy...        0.050000   
6   What is the maximum value for the configured U...        0.305556   
7   What is the HTTP method used for the request o...        0.562500   
8   What clause specifies the details of usage of ...        0.035088   
9   ** What is the parameter that shall be set acc...        0.140845   
10  What is the result of the test specified in th...        0.059701   
11  ** When an FRMCS Equipment in Off-Network mode...        0.076336   
12  What is the value of the "threshx-high" field .

SBERT Similarity (Proximité sémantique)

Elle mesure si deux phrases veulent dire la même chose, même si elles utilisent des mots totalement différents. Elle transforme les phrases en concepts mathématiques pour comparer leur sens profond.
Quand l'utiliser ?
- Moteurs de recherche sémantique : Pour lier la question d'un utilisateur aux documents les plus proches en termes de sens.
- Évaluation de reformulations ou de résumés : Pour vérifier que le texte généré par l'IA conserve l'idée globale du texte d'origine.
Quand l'éviter ? Pour les faits ultra-précis ou les négations. SBERT peut parfois donner un très bon score à deux phrases structurellement proches mais logiquement opposées (ex: "Le patient a pris son traitement" vs "Le patient n'a pas pris son traitement").



In [8]:
import pandas as pd
from sentence_transformers import SentenceTransformer

df = pd.read_csv("evaluation_results.csv")

sbert_model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2"
)


def sbert_similarity(prediction, ground_truth):

    embeddings = sbert_model.encode(
        [prediction, ground_truth],
        normalize_embeddings=True
    )

    score = embeddings[0] @ embeddings[1]

    return float(score)


df["sbert_without_rag"] = df.apply(
    lambda row: sbert_similarity(
        row["llm_without_rag"],
        row["ground_truth"]
    ),
    axis=1
)

df["sbert_with_rag"] = df.apply(
    lambda row: sbert_similarity(
        row["llm_with_rag"],
        row["ground_truth"]
    ),
    axis=1
)

print(df[[
    "question",
    "sbert_without_rag",
    "sbert_with_rag"
]])

                                             question  sbert_without_rag  \
0   ** What are the generic procedure parameters u...           0.449372   
1   If the length of the V2X service info contents...           0.626909   
2   What is the name of the attribute of the media...           0.442074   
3   What is the maximum number of simultaneous MCV...           0.834507   
4   What are the exact instants on which the value...           0.417515   
5   What is the maximum channel bandwidth for a sy...           0.452536   
6   What is the maximum value for the configured U...           0.794175   
7   What is the HTTP method used for the request o...           0.906859   
8   What clause specifies the details of usage of ...           0.220379   
9   ** What is the parameter that shall be set acc...           0.544578   
10  What is the result of the test specified in th...           0.264091   
11  ** When an FRMCS Equipment in Off-Network mode...           0.563609   
12  What is 

 NLI Entailment (Logique et non-hallucination)

Elle vérifie la relation logique entre deux textes. Elle détermine si une phrase de référence (Prémisse) prouve mathématiquement qu'une seconde phrase (Hypothèse) est vraie.

- Détection des hallucinations (Anti-gaspillage/Fiabilité des LLM) : C'est la métrique reine pour s'assurer qu'un chatbot ou un système de résumé automatique n'invente rien. On vérifie si la réponse du LLM est strictement validée par le document source.
- Fact-checking : Pour valider automatiquement une information par rapport à une base de connaissances officielle.
- • Quand l'éviter ? Si vous cherchez juste une ressemblance globale ou un alignement de mots. C'est une métrique purement directionnelle et logique (le texte A implique le texte B, mais l'inverse n'est pas forcément vrai).


In [ ]:
import pandas as pd
import torch

from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

df = pd.read_csv("evaluation_results.csv")

nli_name = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

nli_tokenizer = AutoTokenizer.from_pretrained(nli_name)

nli_model = AutoModelForSequenceClassification.from_pretrained(
    nli_name
)

device = "cuda" if torch.cuda.is_available() else "cpu"

nli_model = nli_model.to(device)


def nli_entailment(prediction, ground_truth):

    inputs = nli_tokenizer(
        ground_truth,
        prediction,
        return_tensors="pt",
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=1
    )[0]

    labels = nli_model.config.id2label

    for index, label in labels.items():

        if "entail" in label.lower():

            return float(
                probabilities[int(index)].cpu()
            )


df["nli_without_rag"] = df.apply(
    lambda row: nli_entailment(
        row["llm_without_rag"],
        row["ground_truth"]
    ),
    axis=1
)

df["nli_with_rag"] = df.apply(
    lambda row: nli_entailment(
        row["llm_with_rag"],
        row["ground_truth"]
    ),
    axis=1
)

print(df[[
    "question",
    "nli_without_rag",
    "nli_with_rag"
]])

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


                                             question  nli_without_rag  \
0   ** What are the generic procedure parameters u...         0.595440   
1   If the length of the V2X service info contents...         0.022371   
2   What is the name of the attribute of the media...         0.153695   
3   What is the maximum number of simultaneous MCV...         0.000357   
4   What are the exact instants on which the value...         0.251575   
5   What is the maximum channel bandwidth for a sy...         0.009051   
6   What is the maximum value for the configured U...         0.001511   
7   What is the HTTP method used for the request o...         0.011098   
8   What clause specifies the details of usage of ...         0.068107   
9   ** What is the parameter that shall be set acc...         0.180171   
10  What is the result of the test specified in th...         0.282844   
11  ** When an FRMCS Equipment in Off-Network mode...         0.993399   
12  What is the value of the "threshx-

Pour sentence-transformers/all-mpnet-base-v2, tu peux dire qu’il est adapté parce que ton objectif est de mesurer la similarité sémantique entre la réponse de Phi-4 et la Ground Truth. Ce modèle est conçu pour produire des embeddings de phrases et de paragraphes afin de comparer leur sens avec une cosine similarity. Il est plus pertinent qu’une comparaison purement lexicale, car deux réponses peuvent être correctes tout en utilisant des mots différents. En plus, il reste assez léger pour être utilisé localement sur plusieurs réponses.


Pour MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli, l’argument principal est qu’il est spécifiquement entraîné pour le Natural Language Inference. Il prédit entailment, neutral ou contradiction, ce qui correspond directement à ton besoin de vérifier si une réponse est réellement supportée par une Ground Truth ou par un contexte RAG. Il a été entraîné sur plusieurs datasets NLI, notamment MNLI, FEVER-NLI et ANLI, ce qui le rend intéressant pour des cas de vérification factuelle et de contradictions plus difficiles.